# RO-048 Grid path planning: every number in the Note

The planners live in `images/gridplan.py`, on chapter RO-04's room grid (`images/searchkit.py`: RO-009's room in 0.5 m
cells, 76 free; start cell S, goal cell G under the door; 8-connected moves of 0.5 m and 0.707 m that never cut the
corner of a blocked cell). This notebook runs each planner and prints the numbers the Note quotes. Nothing here is
random, so there is no seed to set.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
from gridplan import *
from searchkit import astar
r3 = lambda x: round(x, 3)
print('free cells', len(CELLS), ' start', START, ' goal', GOAL)

free cells 76  start (4, 2)  goal (7, 7)


## 1. One route per start: A* again from every cell (Overview, Figure 1)

In [2]:
A = astar()
print('A* from S:', len(A['order']), 'cells, route', r3(A['cost']), 'm', A['path'])
B = astar(start=(5, 2))
print('A* from the slip cell (5, 2):', len(B['order']), 'cells, route', r3(B['cost']), 'm', B['path'])
print('A* from each of the 76 free cells, cells taken off in total:', sum(len(astar(start=c)['order']) for c in CELLS))

A* from S: 9 cells, route 3.414 m [(4, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
A* from the slip cell (5, 2): 7 cells, route 3.207 m [(5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
A* from each of the 76 free cells, cells taken off in total: 902


## 2. The wavefront on the 4-connected grid (Figure 2)

In [3]:
W4, R4 = wavefront(nb4)
print('cells numbered', len(W4), ' S is', W4[START], 'moves =', W4[START] * 0.5, 'm')
print('ring sizes', [len(r) for r in R4])
print('farthest', max(W4.values()), 'moves')

cells numbered 76  S is 8 moves = 4.0 m
ring sizes [1, 3, 4, 4, 4, 4, 4, 4, 5, 7, 7, 6, 5, 4, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1]
farthest 24 moves


## 3. Connectivity and counting metres (Figure 3)

In [4]:
W8, R8 = wavefront(neighbours)
V = cost_to_go()
print('8-connected, counting moves: S is', W8[START], 'moves')
p8a = [(4, 2), (4, 3), (4, 4), (4, 5), (5, 6), (6, 6), (7, 7)]
p8b = [(4, 2), (3, 3), (3, 4), (4, 5), (5, 6), (6, 6), (7, 7)]
for p in (p8a, p8b):
    print('  downhill walk on the moves map:', all(W8[b] == W8[a] - 1 for a, b in zip(p, p[1:])), r3(path_cost(p)), 'm')
def alldown(c, val, nbrs, path=()):
    path = path + (c,)
    if c == GOAL: return [path]
    m = min(val[n] for n, _ in nbrs(c))
    return [q for n, _ in nbrs(c) if val[n] == m for q in alldown(n, val, nbrs, path)]
walks = alldown(START, W8, neighbours)
print('  all downhill walks from S:', len(walks), ' of them 3.828 m:', sum(abs(path_cost(w) - 3.828) < 1e-3 for w in walks))
print('8-connected, counting metres: S is', r3(V[START]), 'm; downhill', downhill(START, V), r3(path_cost(downhill(START, V))))
print('4-connected downhill route', r3(path_cost(downhill(START, W4, nb4))), 'm')

8-connected, counting moves: S is 6 moves
  downhill walk on the moves map: True 3.414 m
  downhill walk on the moves map: True 3.828 m
  all downhill walks from S: 24  of them 3.828 m: 16
8-connected, counting metres: S is 3.414 m; downhill [(4, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)] 3.414
4-connected downhill route 4.0 m


## 4. The navigation function: one minimum, every walk exact (Figures 4 and 5)

In [5]:
nxt = {c: downhill(c, V)[1] for c in CELLS if c != GOAL}
print('every cell but G has a lower neighbour:', all(V[n] < V[c] for c, n in nxt.items()))
print('largest number', r3(max(V.values())))
for c in (START, (5, 2), (0, 0), (13, 0)):
    p = downhill(c, V)
    print(c, 'number', r3(V[c]), ' walk', r3(path_cost(p)), 'm,', len(p) - 1, 'moves', p)
print('walk from S, numbers on the way:', [r3(V[c]) for c in downhill(START, V)])
print('first step from S: move + number')
for n, s in neighbours(START):
    print('  ', n, r3(s), '+', r3(V[n]), '=', r3(s + V[n]))
def lowest(c):
    p = [c]
    while p[-1] != GOAL:
        p.append(min(neighbours(p[-1]), key=lambda ns: (V[ns[0]], ns[0]))[0])
    return p
print('cells where stepping to the lowest neighbour gives a longer walk:',
      sum(1 for c in CELLS if c != GOAL and path_cost(lowest(c)) > V[c] + 1e-9))

every cell but G has a lower neighbour: True
largest number 11.121
(4, 2) number 3.414  walk 3.414 m, 6 moves [(4, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
(5, 2) number 3.207  walk 3.207 m, 6 moves [(5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
(0, 0) number 5.243  walk 5.243 m, 8 moves [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
(13, 0) number 7.914  walk 7.914 m, 15 moves [(13, 0), (12, 0), (11, 0), (10, 0), (9, 1), (8, 1), (7, 1), (6, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 7), (7, 7)]
walk from S, numbers on the way: [3.414, 2.707, 2.207, 1.707, 1.207, 0.5, 0.0]
first step from S: move + number
   (3, 1) 0.707 + 4.121 = 4.828
   (3, 2) 0.5 + 3.621 = 4.121
   (3, 3) 0.707 + 3.121 = 3.828
   (4, 1) 0.5 + 3.914 = 4.414
   (4, 3) 0.5 + 2.914 = 3.414
   (5, 1) 0.707 + 3.707 = 4.414
   (5, 2) 0.5 + 3.207 = 3.707
   (5, 3) 0.707 + 2.707 = 3.414
cells where stepping to the lowest neighbour gives a longer walk: 0


## 5. Value iteration, in place (Figure 6)

In [6]:
SN = value_iteration()
print('sweeps', len(SN) - 1)
for k, s in enumerate(SN[1:], 1):
    print('sweep', k, ' new', len(s['new']), ' lowered', len(s['lowered']), ' S', r3(s['V'][START]),
          {c: (r3(a), r3(b)) for c, (a, b) in s['lowered'].items()})
print('lowered in all', sum(len(s['lowered']) for s in SN))
print('final values equal Dijkstra from the door:', all(abs(SN[-1]['V'][c] - V[c]) < 1e-9 for c in CELLS))
print('cell updates, 8 sweeps x 75 cells:', 8 * 75)
# the same update with each sweep using only the last sweep's values (two arrays)
U = {c: math.inf for c in CELLS}; U[GOAL] = 0.0; k = 0
while True:
    k += 1
    W = {c: 0.0 if c == GOAL else min((s + U[n] for n, s in neighbours(c)), default=math.inf) for c in CELLS}
    if W == U: break
    U = W
print('two-array sweeps until nothing changes:', k)
best = {GOAL: 0}
for c in sorted(CELLS, key=lambda c: V[c])[1:]:
    best[c] = min(best[n] + 1 for n, s in neighbours(c) if abs(V[n] + s - V[c]) < 1e-9)
print('most moves on any shortest route:', max(best.values()), '; S needs', best[START])

sweeps 8
sweep 1  new 49  lowered 0  S 3.414 {}
sweep 2  new 11  lowered 0  S 3.414 {}
sweep 3  new 3  lowered 4  S 3.414 {(4, 7): (1.914, 1.5), (3, 6): (2.621, 2.207), (1, 4): (4.036, 3.621), (0, 3): (4.743, 4.328)}
sweep 4  new 6  lowered 2  S 3.414 {(3, 7): (2.414, 2.0), (0, 4): (4.536, 4.121)}
sweep 5  new 3  lowered 3  S 3.414 {(2, 7): (2.914, 2.5), (0, 5): (5.036, 4.621), (11, 2): (10.328, 9.914)}
sweep 6  new 2  lowered 2  S 3.414 {(1, 7): (3.414, 3.0), (0, 6): (5.536, 4.414)}
sweep 7  new 1  lowered 3  S 3.414 {(0, 7): (3.914, 3.5), (0, 6): (4.414, 4.0), (0, 5): (4.621, 4.5)}
sweep 8  new 0  lowered 0  S 3.414 {}
lowered in all 14
final values equal Dijkstra from the door: True
cell updates, 8 sweeps x 75 cells: 600
two-array sweeps until nothing changes: 22
most moves on any shortest route: 21 ; S needs 6


## 6. Dynamic programming with interpolation (Figures 7 and 8)

In [7]:
for K in (8, 16, 32):
    VI, sw = dp_interp(K)
    pts, L = follow(VI, 2.0, 1.0, K)
    print('K', K, ' sweeps', sw, ' S value', r3(VI[START]), ' route from (2, 1)', r3(L), 'm in', len(pts) - 1, 'moves',
          ' route from S centre', r3(follow(VI, *centre(START), K)[1]))
print('grid way from (2, 1): to S centre', r3(math.hypot(0.25, 0.25)), '+', r3(V[START]), '=', r3(math.hypot(0.25, 0.25) + V[START]))
VI, sw = dp_interp(16)
x, y = 2.0 + 0.5 * math.cos(math.radians(67.5)), 1.0 + 0.5 * math.sin(math.radians(67.5))
print('first move end', (round(x, 3), round(y, 3)))
fx, fy = x / 0.5 - 0.5, y / 0.5 - 0.5
i, j = math.floor(fx), math.floor(fy); ax, ay = fx - i, fy - j
print('across', round(ax, 4), ' up', round(ay, 4))
tot = 0
for c, w in (((i, j), (1 - ax) * (1 - ay)), ((i + 1, j), ax * (1 - ay)), ((i, j + 1), (1 - ax) * ay), ((i + 1, j + 1), ax * ay)):
    print('  ', c, 'value', r3(VI[c]), 'weight', round(w, 4), 'product', round(w * VI[c], 4)); tot += w * VI[c]
print('blend', round(tot, 4), ' check', round(interp(VI, x, y), 4), ' 0.5 + blend', round(0.5 + tot, 4))
print('all 16 moves from (2, 1), 0.5 + value at the end:', [r3(0.5 + interp(VI, 2 + 0.5 * u, 1 + 0.5 * v)) for u, v in directions(16)])

K 8  sweeps 17  S value 3.43  route from (2, 1) 3.768 m in 8 moves  route from S centre 3.414
K 16  sweeps 17  S value 3.338  route from (2, 1) 3.564 m in 8 moves  route from S centre 3.303
K 32  sweeps 17  S value 3.335  route from (2, 1) 3.425 m in 7 moves  route from S centre 3.191
grid way from (2, 1): to S centre 0.354 + 3.414 = 3.768
first move end (2.191, 1.462)
across 0.8827  up 0.4239
   (3, 2) value 3.557 weight 0.0676 product 0.2404
   (4, 2) value 3.338 weight 0.5085 product 1.6976
   (3, 3) value 3.122 weight 0.0497 product 0.1552
   (4, 3) value 2.862 weight 0.3742 product 1.0708
blend 3.1641  check 3.1641  0.5 + blend 3.6641
all 16 moves from (2, 1), 0.5 + value at the end: [4.017, 3.839, 3.711, 3.664, 3.72, 3.845, 4.028, 4.23, 4.419, 4.567, 4.657, 4.686, 4.656, 4.553, 4.401, 4.215]


## 7. Fast marching and the Eikonal equation (Figures 9 and 10)

In [8]:
print('fast marching on the 0.5 m grid itself: S', r3(fmm(*fine_grid(1), GOAL)[0][START[1], START[0]]), ' Dijkstra', r3(V[START]))
F5, H5 = fine_grid(5)
src, s = (37, 37), (22, 12)
TF, _ = fmm(F5, H5, src); TD, _ = dijkstra8(F5, H5, src)
print('0.1 m cells:', int(F5.sum()), ' S reached: fast marching', r3(TF[s[1], s[0]]), ' 8-connected Dijkstra', r3(TD[s[1], s[0]]))
a = min(TF[s[1], s[0] - 1], TF[s[1], s[0] + 1]); b = min(TF[s[1] - 1, s[0]], TF[s[1] + 1, s[0]])
print('at S: a', round(a, 4), ' b', round(b, 4), ' update', round(eikonal_update(a, b, H5), 4))
print('45 degree check: a = b = 1.0, h = 0.1 ->', round(eikonal_update(1.0, 1.0, 0.1), 5), ' exact', round(1 + 0.1 / math.sqrt(2), 5))
from shapely.geometry import box, LineString
from shapely.ops import unary_union
blk = unary_union([box(i * 0.5, j * 0.5, i * 0.5 + 0.5, j * 0.5 + 0.5) for (i, j) in
                   [(i, j) for j in range(8) for i in range(14) if not free((i, j))]])
print('straight line S centre -> (3, 3) -> G centre:', r3(math.dist((2.25, 1.25), (3, 3)) + math.dist((3, 3), (3.75, 3.75))),
      ' each leg clear:', not LineString([(2.25, 1.25), (3, 3)]).crosses(blk), not LineString([(3, 3), (3.75, 3.75)]).crosses(blk))
for k in (5, 10, 20):
    Fk, Hk = fine_grid(k); sk_ = (START[0] * k + k // 2, START[1] * k + k // 2); gk = (GOAL[0] * k + k // 2, GOAL[1] * k + k // 2)
    print('cells', Hk, 'm: fast marching', r3(fmm(Fk, Hk, gk)[0][sk_[1], sk_[0]]), ' Dijkstra', r3(dijkstra8(Fk, Hk, gk)[0][sk_[1], sk_[0]]))
print('open floor, a point 4 m across and 2 m up from the start, exact', r3(math.hypot(4, 2)))
for h, n in ((0.5, 9), (0.1, 41), (0.05, 81), (0.025, 161)):
    fr = np.ones((n, n), bool); Tf = fmm(fr, h, (0, 0))[0]; Td = dijkstra8(fr, h, (0, 0))[0]
    i, j = n - 1, (n - 1) // 2; ex = h * math.hypot(i, j)
    print(' cell', h, 'm: fast marching', r3(Tf[j, i]), f'({(Tf[j, i] / ex - 1) * 100:.1f} percent long)',
          ' Dijkstra', r3(Td[j, i]), f'({(Td[j, i] / ex - 1) * 100:.1f} percent long)')

fast marching on the 0.5 m grid itself: S 3.45  Dijkstra 3.414
0.1 m cells: 1900  S reached: fast marching 3.121  8-connected Dijkstra 3.18
at S: a 3.0807  b 3.0288  update 3.1205
45 degree check: a = b = 1.0, h = 0.1 -> 1.07071  exact 1.07071


straight line S centre -> (3, 3) -> G centre: 2.965  each leg clear: True True


cells 0.1 m: fast marching 3.121  Dijkstra 3.18


cells 0.05 m: fast marching 3.061  Dijkstra 3.151


cells 0.025 m: fast marching 3.021  Dijkstra 3.136
open floor, a point 4 m across and 2 m up from the start, exact 4.472
 cell 0.5 m: fast marching 4.751 (6.2 percent long)  Dijkstra 4.828 (8.0 percent long)
 cell 0.1 m: fast marching 4.563 (2.0 percent long)  Dijkstra 4.828 (8.0 percent long)
 cell 0.05 m: fast marching 4.526 (1.2 percent long)  Dijkstra 4.828 (8.0 percent long)


 cell 0.025 m: fast marching 4.503 (0.7 percent long)  Dijkstra 4.828 (8.0 percent long)
